In [ ]:
from pathlib import Path


TRAIN_CONFIG_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/"
    "radiology/mrsbrain/public/hfish/walinet/"
    "configs/Training/VisulizationNotebook.yaml"
)

SEED = 123456

N_EXAMPLES = 100_000
BATCH_SIZE = 4096

In [ ]:
from pathlib import Path
import sys


project_root = Path.cwd().resolve()

while not (
    project_root
    / "src"
    / "walinet"
).is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "WALINET-Projektordner nicht gefunden."
        )

    project_root = project_root.parent


src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(
        0,
        str(src_dir),
    )


print(
    "WALINET source:",
    src_dir,
)


from walinet.visualization.VisualizeSim import (
    plot_spectra_range,
)

In [ ]:
import torch

from walinet.training_data.build_simulation_system import (
    build_simulation_system,
)


system = build_simulation_system(
    TRAIN_CONFIG_PATH
)

train_cfg = system.train_config
simulation_cfg = system.simulation_config
resources = system.resources
pool = resources.train

prepared_basis = system.prepared_basis
metabolite_simulator = system.metabolite_simulator
spectrum_simulator = system.train_simulator
device = system.device


generator = torch.Generator(
    device=device
)

generator.manual_seed(
    SEED
)


print(
    "Simulation system ready."
)

print(
    "Device:",
    device,
)

print(
    "Simulation config:",
    system.simulation_config_path,
)

print(
    "Basis library:",
    simulation_cfg.basis.library,
)

print(
    "Training subjects:",
    pool.subject_names,
)

In [ ]:
import torch
import torch.nn.functional as F

n_timepoints = spectrum_simulator.n_timepoints


def right_pad_spectra(
    spectra: torch.Tensor,
    target_n_timepoints: int,
) -> torch.Tensor:
    current_n_timepoints = spectra.shape[-1]

    if current_n_timepoints > target_n_timepoints:
        raise ValueError(
            f"Spectrum has {current_n_timepoints} points, "
            f"but target length is only {target_n_timepoints}."
        )

    if current_n_timepoints == target_n_timepoints:
        return spectra

    # Reine Kontrolldarstellung im Frequenzraum: Das native
    # Spektrum bleibt unverändert und wird rechts mit Nullen
    # auf die maximale Länge aufgefüllt.
    return F.pad(
        spectra,
        (
            0,
            target_n_timepoints
            - current_n_timepoints,
        ),
        mode="constant",
        value=0,
    )

# Große Ergebnistensoren auf der CPU speichern.
total_spectra = torch.empty(
    (N_EXAMPLES, n_timepoints),
    dtype=torch.complex64,
    device="cpu",
)

baseline_spectra = torch.empty(
    (N_EXAMPLES, n_timepoints),
    dtype=torch.complex64,
    device="cpu",
)

acquired_lengths = torch.empty(
    (N_EXAMPLES,),
    dtype=torch.int64,
    device="cpu",
)

generated = 0

while generated < N_EXAMPLES:
    current_batch_size = min(
        BATCH_SIZE,
        N_EXAMPLES - generated,
    )

    batch = spectrum_simulator.simulate(
        batch_size=current_batch_size,
        generator=generator,
    )

    end = generated + current_batch_size

    input_spectra = right_pad_spectra(
        batch.normalized_input_spectra,
        n_timepoints,
    ).cpu()

    target_spectra = right_pad_spectra(
        batch.normalized_target_spectra,
        n_timepoints,
    ).cpu()

    total_spectra[generated:end].copy_(
        input_spectra
    )

    baseline_spectra[generated:end].copy_(
        target_spectra
    )

    acquired_lengths[generated:end].copy_(
        batch.acquired_n_timepoints.cpu()
    )

    generated = end

    if generated % 100_000 < BATCH_SIZE:
        print(f"{generated:,} / {N_EXAMPLES:,}")

    del batch

print("Finished")
print("total_spectra:", total_spectra.shape)
print("baseline_spectra:", baseline_spectra.shape)

metabolite_spectra = (
    total_spectra
    - baseline_spectra
)

In [ ]:
import torch
import matplotlib.pyplot as plt


# Prüfen, dass im Frequenzraum ab der nativen Länge wirklich
# ausschließlich exakt gesetzte Nullen stehen.
indices = torch.arange(n_timepoints)[None, :]
padding_mask = indices >= acquired_lengths[:, None]

total_padding_is_zero = torch.all(
    total_spectra[padding_mask] == 0
)
baseline_padding_is_zero = torch.all(
    baseline_spectra[padding_mask] == 0
)

print(
    "Simulierte native Längen:",
    acquired_lengths.min().item(),
    "bis",
    acquired_lengths.max().item(),
)
print(
    "Total rechts exakt null:",
    total_padding_is_zero.item(),
)
print(
    "Baseline rechts exakt null:",
    baseline_padding_is_zero.item(),
)
print(
    "Erste 20 Längen:",
    acquired_lengths[:20].tolist(),
)

In [ ]:
ratio = (
    torch.amax(torch.abs(metabolite_spectra), dim=-1)
    / torch.amax(torch.abs(total_spectra), dim=-1)
)

print("Minimum ratio:", ratio.max().item())
print("Index:", ratio.argmin().item())

In [ ]:
plot_spectra_range(
    total_spectra=total_spectra,
    metabolite_spectra=metabolite_spectra,
    simulation_cfg=simulation_cfg,
    start=0,
    stop=20,
    component="real",
    ppm_min=0.0,
    ppm_max=7.0,
)

In [ ]:
import time
import torch


N_EXAMPLES = 1_500_000
BATCH_SIZE = 4096

generated = 0

torch.cuda.synchronize()
start = time.perf_counter()

while generated < N_EXAMPLES:
    current_batch_size = min(
        BATCH_SIZE,
        N_EXAMPLES - generated,
    )

    batch = spectrum_simulator.simulate(
        batch_size=current_batch_size,
        generator=generator,
    )

    generated += current_batch_size
    del batch

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(f"Zeit: {elapsed:.2f} s")
print(f"Rate: {N_EXAMPLES / elapsed:,.0f} Spektren/s")